# Chess Learning — Stage 01: Engine

This notebook implements the core chess engine components:

| Component | Description |
|---|---|
| **Feature extractor** | Converts a board position to a named feature vector |
| **Linear evaluator** | Weighted dot product of features → position score |
| **Negamax search** | Alpha-beta pruning → best move |

All features are computed from the **perspective of the side to move** (positive = good for the side to move). This lets the negamax search simply negate scores at each ply.

Compatible with Google Colab and Positron.

## Setup

In [ ]:
# Install python-chess (safe to run even if already installed)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'chess', '-q'])

In [ ]:
import chess
import random
from typing import Optional

## Feature Registry

The feature names define the full set of chess concepts being tracked. Each name maps to a weight in the evaluator. Weight trajectories over training are the primary research data.

| Feature | What it captures |
|---|---|
| `material_*` | Piece count advantage by type |
| `passed_pawn` | Pawns with no opposing pawns ahead |
| `doubled_pawn` | Extra pawns on same file (penalty) |
| `isolated_pawn` | Pawns with no friendly neighbours (penalty) |
| `king_safety` | Safe squares adjacent to king |
| `center_control` | Attacks on d4/d5/e4/e5 |
| `rook_open_file` | Rooks on pawn-free files |
| `connected_rooks` | Both rooks on same rank/file, unobstructed |

Penalty features are negated so a **positive weight is always beneficial** for the side to move.

In [ ]:
FEATURE_NAMES = [
    # Material
    'material_pawn',
    'material_knight',
    'material_bishop',
    'material_rook',
    'material_queen',
    # Pawn structure
    'passed_pawn',
    'doubled_pawn',
    'isolated_pawn',
    # King safety
    'king_safety',
    # Activity
    'center_control',
    'rook_open_file',
    'connected_rooks',
]


def default_weights(init: str = 'zero') -> dict:
    # init='zero'   -> learn from scratch
    # init='random' -> small random values to break symmetry
    if init == 'zero':
        return {name: 0.0 for name in FEATURE_NAMES}
    elif init == 'random':
        return {name: random.uniform(-0.1, 0.1) for name in FEATURE_NAMES}
    raise ValueError(f'Unknown init mode: {init!r}')

## Feature Extraction Helpers

Each helper computes a single concept for one color. The main extractor calls each pair (our color, their color) and takes the difference.

In [ ]:
def _count_passed_pawns(board: chess.Board, color: chess.Color) -> int:
    # Pawns with no opposing pawns on the same or adjacent files ahead
    our_pawns = board.pieces(chess.PAWN, color)
    opp_pawns = [
        (chess.square_file(sq), chess.square_rank(sq))
        for sq in board.pieces(chess.PAWN, not color)
    ]
    count = 0
    for sq in our_pawns:
        file = chess.square_file(sq)
        rank = chess.square_rank(sq)
        adj = [f for f in (file - 1, file, file + 1) if 0 <= f <= 7]
        passed = True
        for opp_f, opp_r in opp_pawns:
            if opp_f in adj:
                if color == chess.WHITE and opp_r > rank:
                    passed = False; break
                elif color == chess.BLACK and opp_r < rank:
                    passed = False; break
        if passed:
            count += 1
    return count


def _count_doubled_pawns(board: chess.Board, color: chess.Color) -> int:
    # Extra pawns beyond one on any file
    file_counts: dict[int, int] = {}
    for sq in board.pieces(chess.PAWN, color):
        f = chess.square_file(sq)
        file_counts[f] = file_counts.get(f, 0) + 1
    return sum(max(0, n - 1) for n in file_counts.values())


def _count_isolated_pawns(board: chess.Board, color: chess.Color) -> int:
    # Pawns with no friendly pawns on adjacent files
    our_pawns = board.pieces(chess.PAWN, color)
    pawn_files = {chess.square_file(sq) for sq in our_pawns}
    return sum(
        1 for sq in our_pawns
        if not ({chess.square_file(sq) - 1, chess.square_file(sq) + 1} & pawn_files)
    )


def _king_safety_score(board: chess.Board, color: chess.Color) -> int:
    # Safe squares adjacent to king (not attacked by opponent)
    king_sq = board.king(color)
    if king_sq is None:
        return 0
    opp = not color
    return sum(
        1 for sq in chess.SquareSet(chess.BB_KING_ATTACKS[king_sq])
        if not board.is_attacked_by(opp, sq)
    )


def _center_control_score(board: chess.Board, color: chess.Color) -> int:
    # Central squares attacked by color
    center = (chess.D4, chess.D5, chess.E4, chess.E5)
    return sum(1 for sq in center if board.is_attacked_by(color, sq))


def _count_rooks_open_file(board: chess.Board, color: chess.Color) -> int:
    # Rooks on files with no pawns of either color
    all_pawn_files = {
        chess.square_file(sq)
        for sq in board.pieces(chess.PAWN, chess.WHITE) | board.pieces(chess.PAWN, chess.BLACK)
    }
    return sum(
        1 for sq in board.pieces(chess.ROOK, color)
        if chess.square_file(sq) not in all_pawn_files
    )


def _are_rooks_connected(board: chess.Board, color: chess.Color) -> bool:
    # Both rooks on same rank or file with no pieces between them
    rooks = list(board.pieces(chess.ROOK, color))
    if len(rooks) < 2:
        return False
    r1, r2 = rooks[0], rooks[1]
    f1, rank1 = chess.square_file(r1), chess.square_rank(r1)
    f2, rank2 = chess.square_file(r2), chess.square_rank(r2)
    if f1 == f2:
        lo, hi = sorted((rank1, rank2))
        return all(board.piece_at(chess.square(f1, r)) is None for r in range(lo + 1, hi))
    if rank1 == rank2:
        lo, hi = sorted((f1, f2))
        return all(board.piece_at(chess.square(f, rank1)) is None for f in range(lo + 1, hi))
    return False

## Feature Extractor

`extract_features` calls each helper pair and returns a single dict. The sign convention ensures a **positive weight is always good** for the side to move — including penalty features like `doubled_pawn`, which are negated.

In [ ]:
def extract_features(board: chess.Board) -> dict:
    us   = board.turn
    them = not us
    return {
        # Material: our count minus opponent's count
        'material_pawn':    len(board.pieces(chess.PAWN,   us)) - len(board.pieces(chess.PAWN,   them)),
        'material_knight':  len(board.pieces(chess.KNIGHT, us)) - len(board.pieces(chess.KNIGHT, them)),
        'material_bishop':  len(board.pieces(chess.BISHOP, us)) - len(board.pieces(chess.BISHOP, them)),
        'material_rook':    len(board.pieces(chess.ROOK,   us)) - len(board.pieces(chess.ROOK,   them)),
        'material_queen':   len(board.pieces(chess.QUEEN,  us)) - len(board.pieces(chess.QUEEN,  them)),
        # Pawn structure (penalties negated so positive weight = good)
        'passed_pawn':   _count_passed_pawns(board, us)    - _count_passed_pawns(board, them),
        'doubled_pawn':  -(_count_doubled_pawns(board, us) - _count_doubled_pawns(board, them)),
        'isolated_pawn': -(_count_isolated_pawns(board, us)- _count_isolated_pawns(board, them)),
        # King safety and activity
        'king_safety':    _king_safety_score(board, us)    - _king_safety_score(board, them),
        'center_control': _center_control_score(board, us) - _center_control_score(board, them),
        'rook_open_file': _count_rooks_open_file(board, us)- _count_rooks_open_file(board, them),
        'connected_rooks':int(_are_rooks_connected(board, us)) - int(_are_rooks_connected(board, them)),
    }

## Evaluator

A weighted dot product of the feature vector. Terminal states (checkmate, draw) return fixed values and bypass feature computation.

In [ ]:
def evaluate(board: chess.Board, weights: dict) -> float:
    if board.is_checkmate():
        return -10_000.0   # side to move has been mated
    if board.is_stalemate() or board.is_insufficient_material() or board.can_claim_draw():
        return 0.0
    features = extract_features(board)
    return sum(weights.get(name, 0.0) * val for name, val in features.items())

## Search: Negamax with Alpha-Beta Pruning

**Negamax** is a simplified form of minimax that exploits the zero-sum property: the score for one side is the negative of the score for the other. At each node, we negate and recurse — no separate min/max logic needed.

**Alpha-beta pruning** eliminates branches that cannot affect the final decision, roughly doubling the effective search depth for the same computation.

In [ ]:
def negamax(
    board: chess.Board,
    depth: int,
    alpha: float,
    beta: float,
    weights: dict,
) -> float:
    if board.is_game_over():
        return -10_000.0 if board.is_checkmate() else 0.0
    if depth == 0:
        return evaluate(board, weights)
    max_score = float('-inf')
    for move in board.legal_moves:
        board.push(move)
        score = -negamax(board, depth - 1, -beta, -alpha, weights)
        board.pop()
        if score > max_score:
            max_score = score
        if score > alpha:
            alpha = score
        if alpha >= beta:
            break  # beta cutoff
    return max_score


def get_best_move(
    board: chess.Board,
    weights: dict,
    depth: int = 2,
    randomize: bool = True,
) -> tuple[Optional[chess.Move], float]:
    moves = list(board.legal_moves)
    if not moves:
        return None, 0.0
    if randomize:
        # Shuffle to break ties randomly — prevents repetitive games
        # when weights are near zero
        random.shuffle(moves)
    best_move, best_score = None, float('-inf')
    alpha, beta = float('-inf'), float('inf')
    for move in moves:
        board.push(move)
        score = -negamax(board, depth - 1, -beta, -alpha, weights)
        board.pop()
        if score > best_score:
            best_score = score
            best_move  = move
        if score > alpha:
            alpha = score
    return best_move, best_score

## Verification

A set of quick checks to confirm the engine components are working correctly. All cells below should run without errors.

### Feature extraction

In [ ]:
board = chess.Board()
features = extract_features(board)

# All feature names present
assert set(features.keys()) == set(FEATURE_NAMES), 'Missing or extra feature keys'

# Starting position is symmetric: all material features == 0
for name in ['material_pawn', 'material_knight', 'material_bishop', 'material_rook', 'material_queen']:
    assert features[name] == 0, f'{name} should be 0 at start'

print('Feature extraction: OK')
print('Features at starting position:')
for k, v in features.items():
    print(f'  {k}: {v}')

### Material advantage detection

In [ ]:
# White has two queens; black has none (besides kings)
board = chess.Board('4k3/8/8/8/8/8/3Q4/3QK3 w - - 0 1')
features = extract_features(board)
assert features['material_queen'] == 2, f'Expected +2, got {features["material_queen"]}'
print(f'material_queen advantage: {features["material_queen"]} (expected 2) — OK')

### Passed pawn detection

In [ ]:
# White pawn on e5, no black pawns: should be passed
board = chess.Board('4k3/8/8/4P3/8/8/8/4K3 w - - 0 1')
features = extract_features(board)
assert features['passed_pawn'] > 0, f'Expected passed_pawn > 0, got {features["passed_pawn"]}'
print(f'passed_pawn: {features["passed_pawn"]} (expected > 0) — OK')

### Evaluator: terminal states

In [ ]:
# Scholar's mate — black is checkmated
board = chess.Board('r1bqkb1r/pppp1Qpp/2n2n2/4p3/2B1P3/8/PPPP1PPP/RNB1K1NR b KQkq - 0 4')
assert board.is_checkmate()
score = evaluate(board, default_weights())
assert score == -10_000.0, f'Expected -10000, got {score}'
print(f'Checkmate evaluation: {score} — OK')

# Stalemate
board = chess.Board('k7/2Q5/1K6/8/8/8/8/8 b - - 0 1')
assert board.is_stalemate()
score = evaluate(board, default_weights())
assert score == 0.0
print(f'Stalemate evaluation: {score} — OK')

### Search: capture a free queen

In [ ]:
# White pawn on e4, black queen on d5 — pawn captures diagonally
board = chess.Board('4k3/8/8/3q4/4P3/8/8/4K3 w - - 0 1')

w = default_weights()
w['material_pawn']  = 1.0
w['material_queen'] = 9.0

move, score = get_best_move(board, w, depth=1, randomize=False)
assert move == chess.Move.from_uci('e4d5'), f'Expected e4d5, got {move}'
assert score > 0
print(f'Captured free queen: {move}, score={score} — OK')

### Search: find checkmate in one

In [ ]:
# White queen on g7 + rook on h1 vs lone black king on h8.
# Rh7# is checkmate (queen protects the rook; king can't escape).
# The engine may find any mating move — we just verify the result.
board = chess.Board('7k/6Q1/8/5K2/8/8/8/7R w - - 0 1')

move, score = get_best_move(board, default_weights(), depth=1, randomize=False)
board.push(move)
assert board.is_checkmate(), f'Expected checkmate after {move}, was not'
print(f'Checkmate in one: {move}, score={score} — OK')

---
All checks passed. The engine is ready for Stage 02: self-play and TD learning.